## 1.Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

## 2.Load Dataset

In [2]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print(movies.head())
print(ratings.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating   timestamp
0       1       16     4.0  1217897793
1       1       24     1.5  1217895807
2       1       32     4.0  1217896246
3       1       47     4.0  1217896556
4       1       50     4.0  1217896523


## 3.Explore Dataset

In [3]:
print("Movies Shape:", movies.shape)
print("Ratings Shape:", ratings.shape)

movies.info()
ratings.info()

Movies Shape: (10329, 3)
Ratings Shape: (105339, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10329 entries, 0 to 10328
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  10329 non-null  int64 
 1   title    10329 non-null  object
 2   genres   10329 non-null  object
dtypes: int64(1), object(2)
memory usage: 242.2+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105339 entries, 0 to 105338
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     105339 non-null  int64  
 1   movieId    105339 non-null  int64  
 2   rating     105339 non-null  float64
 3   timestamp  105339 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.2 MB


## 4.Content-Based Recommendation System

# Create Features

In [4]:
movies['genres'] = movies['genres'].fillna('')

# TF-IDF Vectorization

In [5]:
tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(movies['genres'])

print(tfidf_matrix.shape)

(10329, 23)


# Cosine Similarity Matrix

In [6]:
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cosine_sim.shape)

(10329, 10329)


# Create Movie Index Mapping

In [7]:
indices = pd.Series(
    movies.index,
    index=movies['title']
).drop_duplicates()

# Recommendation Function

In [9]:
def content_recommend(movie_title, top_n=10):

    if movie_title not in indices:
        return "Movie not found"

    idx = indices[movie_title]

    sim_scores = list(
        enumerate(cosine_sim[idx])
    )

    sim_scores = sorted(
        sim_scores,
        key=lambda x: x[1],
        reverse=True
    )

    sim_scores = sim_scores[1:top_n+1]

    movie_indices = [
        i[0] for i in sim_scores
    ]

    return movies['title'].iloc[
        movie_indices
    ]

# Test Content-Based Recommendation

In [10]:
content_recommend("Toy Story (1995)")

1815                                          Antz (1998)
2496                                   Toy Story 2 (1999)
2967       Adventures of Rocky and Bullwinkle, The (2000)
3166                     Emperor's New Groove, The (2000)
3811                                Monsters, Inc. (2001)
6617    DuckTales: The Movie - Treasure of the Lost La...
6997                                     Wild, The (2006)
7382                               Shrek the Third (2007)
7987                       Tale of Despereaux, The (2008)
9215    Asterix and the Vikings (Astérix et les Viking...
Name: title, dtype: object

## 5.Collaborative Filtering

# Create User-Movie Matrix

In [11]:
user_movie_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

user_movie_matrix = user_movie_matrix.fillna(0)

user_movie_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,144482,144656,144976,146344,146656,146684,146878,148238,148626,149532
userId,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5.0,0.0,2.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,3.0,0.0,3.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# User Similarity

In [12]:
user_similarity = cosine_similarity(
    user_movie_matrix
)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

# Find Similar Users

In [13]:
def get_similar_users(user_id, n=5):

    similar_users = user_similarity_df[user_id]\
        .sort_values(ascending=False)

    return similar_users[1:n+1]

## 6. User-Based Recommendation Function

In [15]:
def collaborative_recommend(user_id, n_movies=10):

    similar_users = get_similar_users(user_id, 5)

    similar_user_ids = similar_users.index

    movies_watched = ratings[
        ratings['userId'] == user_id
    ]['movieId']

    recommended = ratings[
        ratings['userId'].isin(similar_user_ids)
    ]

    recommended = recommended[
        ~recommended['movieId'].isin(movies_watched)
    ]

    movie_scores = recommended.groupby(
        'movieId'
    )['rating'].mean()

    top_movies = movie_scores.sort_values(
        ascending=False
    ).head(n_movies)

    return movies[
        movies['movieId'].isin(top_movies.index)
    ][['title']]

## 7. Hybrid Recommendation System

In [16]:
def hybrid_recommend(user_id,
                     favorite_movie,
                     top_n=10):

    content_movies = content_recommend(
        favorite_movie,
        top_n=20
    )

    collaborative_movies = collaborative_recommend(
        user_id,
        n_movies=20
    )

    hybrid = pd.concat([
        pd.DataFrame(content_movies),
        collaborative_movies
    ])

    hybrid.columns = ['title']

    hybrid = hybrid.drop_duplicates()

    return hybrid.head(top_n)

## 9. Movie Search Function

In [17]:
def search_movie(keyword):

    result = movies[
        movies['title']
        .str.contains(keyword,
                      case=False,
                      na=False)
    ]

    return result[['title']].head(20)

## 9. User Interface

In [21]:
movie_name = input("Enter movie name: ")

recommendations = content_recommend(movie_name)

for i, movie in enumerate(recommendations, 1):
    print(f"{i}. {movie}")

1. Dark Knight, The (2008)
2. Need for Speed (2014)
3. Eagle Eye (2008)
4. Good Day to Die Hard, A (2013)
5. Fast & Furious 6 (Fast and the Furious 6, The) (2013)
6. Fast Five (Fast and the Furious 5, The) (2011)
7. Dark Knight Rises, The (2012)
8. Man of Tai Chi (2013)
9. Grandmaster, The (Yi dai zong shi) (2013)
10. Night at the Museum: Battle of the Smithsonian (2009)
